# 2. Few-Shot Prompting - Aprendizaje con Ejemplos

## Objetivos de Aprendizaje
- Comprender los principios del few-shot prompting
- Seleccionar y estructurar ejemplos efectivos
- Implementar patrones de entrada-salida consistentes
- Optimizar el número y calidad de ejemplos

## ¿Qué es Few-Shot Prompting?

Few-shot prompting es una técnica donde proporcionamos **ejemplos específicos** de la tarea que queremos que el modelo realice. El modelo aprende el patrón de estos ejemplos y lo aplica a nuevas entradas.

### Estructura Básica:
```
Instrucción (opcional)
Ejemplo 1: Input → Output
Ejemplo 2: Input → Output
Ejemplo 3: Input → Output
Nueva entrada: Input → ?
```

### Ventajas:
- **Consistencia**: Resultados más predecibles
- **Formato controlado**: Salidas estructuradas
- **Menos ambigüedad**: Patrones claros a seguir
- **Tareas específicas**: Excelente para casos especializados

### Consideraciones:
- **Más tokens**: Consume más espacio de contexto
- **Selección de ejemplos**: Crucial para el éxito
- **Sesgos**: Los ejemplos pueden introducir sesgos
- **Overfitting**: Puede ser demasiado específico

In [ ]:
import os

os.environ["GITHUB_TOKEN"] = ""
os.environ["OPENAI_BASE_URL"] = ""
os.environ["GROQ_API_KEY"] = ""


In [2]:
# Importar las bibliotecas necesarias
from groq import Groq
from dotenv import load_dotenv
from openai import OpenAI
import os

# load .env
load_dotenv()


# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)

print(os.getenv("GROQ_API_KEY"))


OpenAI library version: 2.26.0
Python version: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
[REDACTED]


In [3]:
# Configuración inicial
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import os
import json

# Configurar el modelo
#llm = ChatOpenAI(
#    base_url=os.getenv("OPENAI_BASE_URL"),
#    api_key=os.getenv("GITHUB_TOKEN"),
#    model="gpt-4o",
#    temperature=0.3  # Menor temperatura para más consistencia
#)

print("✓ Modelo configurado para few-shot prompting")
print("✓ Temperature reducida para mayor consistencia")

✓ Modelo configurado para few-shot prompting
✓ Temperature reducida para mayor consistencia


In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

# Initialize Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant",
    temperature=0.7
)

llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002411155DFD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000241115FD970>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

## Comparación: Zero-Shot vs Few-Shot

Veamos la diferencia en resultados entre ambos enfoques.

In [5]:
# Comparación directa entre enfoques
def comparar_zero_vs_few_shot():
    print("=== COMPARACIÓN: ZERO-SHOT vs FEW-SHOT ===")
    
    # Tarea: Clasificar sentimientos de reviews de productos
    nueva_review = "El producto llegó rápido pero la calidad no es lo que esperaba por el precio."
    
    # Enfoque Zero-Shot
    prompt_zero_shot = f"""Clasifica el sentimiento de esta review como Positivo, Negativo o Neutral:
    
Review: "{nueva_review}"
Sentimiento:"""
    
    # Enfoque Few-Shot
    prompt_few_shot = f"""Clasifica el sentimiento de cada review como Positivo, Negativo o Neutral:
    
Review: "Excelente producto, superó mis expectativas. Muy recomendado."
Sentimiento: Positivo
    
Review: "Llegó defectuoso y el servicio al cliente fue terrible."
Sentimiento: Negativo
    
Review: "Está bien, cumple su función pero nada especial."
Sentimiento: Neutral
    
Review: "Buena relación calidad-precio, aunque podría mejorar el diseño."
Sentimiento: Positivo
    
Review: "{nueva_review}"
Sentimiento:"""
    
    # Probar ambos enfoques
    print("\n1. ZERO-SHOT:")
    print("-" * 15)
    try:
        response_zero = llm.invoke([HumanMessage(content=prompt_zero_shot)])
        print(f"Resultado: {response_zero.content.strip()}")
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n2. FEW-SHOT:")
    print("-" * 15)
    try:
        response_few = llm.invoke([HumanMessage(content=prompt_few_shot)])
        print(f"Resultado: {response_few.content.strip()}")
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n=== ANÁLISIS ===")
    print("• Zero-shot: Puede ser menos consistente en formato")
    print("• Few-shot: Más probable que siga el patrón exacto")
    print("• Few-shot: Mejor para tareas con formato específico")
    
    # Análisis de tokens
    tokens_zero = len(prompt_zero_shot.split())
    tokens_few = len(prompt_few_shot.split())
    print(f"\n• Tokens zero-shot: ~{tokens_zero}")
    print(f"• Tokens few-shot: ~{tokens_few} ({tokens_few/tokens_zero:.1f}x más)")

# Ejecutar comparación
comparar_zero_vs_few_shot()

=== COMPARACIÓN: ZERO-SHOT vs FEW-SHOT ===

1. ZERO-SHOT:
---------------
Resultado: El sentimiento de esta review es: Negativo.

La razón es que aunque el reviewer menciona que el producto llegó rápido, que es un aspecto positivo, también critica la calidad del producto, lo que es un aspecto negativo. La crítica sobre la calidad y el precio supera la satisfacción con el tiempo de entrega, lo que da un tono global negativo a la review.

2. FEW-SHOT:
---------------
Resultado: El sentimiento de la última review es:

Sentimiento: Negativo

Aunque la persona menciona que el producto llegó rápido, lo que podría considerarse positivo, su principal queja es que la calidad no es lo que esperaba por el precio, lo que se vuelve negativo.

=== ANÁLISIS ===
• Zero-shot: Puede ser menos consistente en formato
• Few-shot: Más probable que siga el patrón exacto
• Few-shot: Mejor para tareas con formato específico

• Tokens zero-shot: ~28
• Tokens few-shot: ~72 (2.6x más)


## Selección Estratégica de Ejemplos

La calidad y selección de ejemplos es crucial para el éxito del few-shot prompting.

In [6]:
# Técnica 1: Ejemplos Representativos
def ejemplos_representativos():
    print("=== SELECCIÓN DE EJEMPLOS REPRESENTATIVOS ===")
    
    # Tarea: Extraer información de contacto de emails
    nuevo_email = """Hola, soy María García, gerente de ventas en TechCorp. 
    Mi teléfono es +34 678 901 234 y mi email corporativo es m.garcia@techcorp.es. 
    Nos gustaría agendar una reunión para discutir nuestra propuesta."""
    
    # Buenos ejemplos: diversos y representativos
    prompt_buenos_ejemplos = f"""Extrae información de contacto de cada email en formato JSON:
    
Email: "Saludos, soy Dr. Pedro López del Hospital Central. Pueden contactarme al 915-555-0123 o p.lopez@hospital.com para cualquier consulta médica."
JSON: {{"nombre": "Dr. Pedro López", "empresa": "Hospital Central", "telefono": "915-555-0123", "email": "p.lopez@hospital.com", "cargo": "Doctor"}}
    
Email: "Ana Ruiz, desarrolladora senior en StartupXYZ. Mi número directo es 661-234-567 y mi correo personal es ana.ruiz.dev@gmail.com"
JSON: {{"nombre": "Ana Ruiz", "empresa": "StartupXYZ", "telefono": "661-234-567", "email": "ana.ruiz.dev@gmail.com", "cargo": "Desarrolladora Senior"}}
    
Email: "Contacto comercial: Luis Martín, sin empresa específica. Tel: +1-555-0199, email: luis.martin.comercial@outlook.com"
JSON: {{"nombre": "Luis Martín", "empresa": null, "telefono": "+1-555-0199", "email": "luis.martin.comercial@outlook.com", "cargo": "Comercial"}}
    
Email: "{nuevo_email}"
JSON:

        Tu resultados debe estar en el siguiente formato:
        {{"nombre": "Dr. Pedro López", "empresa": "Hospital Central", "telefono": "915-555-0123", "email": "p.lopez@hospital.com", "cargo": "Doctor"}}
        Tu resultados siempre debe ser un JSON. No pongas ninguna palabra antes, ni despues, solo el JSON. No pongas un formato asi:
        ""json{{"nombre": "Dr. Pedro López", "empresa": "Hospital Central", "telefono": "915-555-0123", "email": "p.lopez@hospital.com", "cargo": "Doctor"}}""
        NUNCA pongas la palabra JSON. 
        Responde ÚNICAMENTE con JSON válido. Los valores de texto DEBEN estar entre comillas dobles.
        Dame solo el JSON del nuevo mensaje, no los demas. 
        """
    
    print("USANDO EJEMPLOS REPRESENTATIVOS:")
    try:
        response = llm.invoke([HumanMessage(content=prompt_buenos_ejemplos)])
        print("Resultado:")
        print(response.content)
        
        # Intentar parsear JSON
        try:
            result_json = json.loads(response.content.strip())
            print("\n✓ JSON válido generado")
            print(f"✓ Campos extraídos: {list(result_json.keys())}")
        except:
            print("\n✗ JSON inválido - formato inconsistente")
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar selección de ejemplos
ejemplos_representativos()

=== SELECCIÓN DE EJEMPLOS REPRESENTATIVOS ===
USANDO EJEMPLOS REPRESENTATIVOS:
Resultado:
{"nombre": "María García", "empresa": "TechCorp", "telefono": "+34 678 901 234", "email": "m.garcia@techcorp.es", "cargo": "Gerente de Ventas"}

✓ JSON válido generado
✓ Campos extraídos: ['nombre', 'empresa', 'telefono', 'email', 'cargo']


In [7]:
# Técnica 2: Balanceo de Categorías
def balanceo_categorias():
    print("=== BALANCEO DE CATEGORÍAS ===")
    
    # Tarea: Clasificar tickets de soporte
    nuevo_ticket = "Mi aplicación se cierra inesperadamente cuando intento abrir archivos grandes. ¿Pueden ayudarme?"
    
    # Ejemplos balanceados por categoría
    prompt_balanceado = f"""Clasifica cada ticket de soporte en: TÉCNICO, FACTURACIÓN, GENERAL:
    
Ticket: "No puedo acceder a mi cuenta, dice que mi contraseña es incorrecta"
Categoría: TÉCNICO
    
Ticket: "¿Cuándo se procesará mi reembolso del mes pasado?"
Categoría: FACTURACIÓN
    
Ticket: "¿Tienen planes de expandirse a otros países?"
Categoría: GENERAL
    
Ticket: "El botón de exportar datos no funciona en Chrome"
Categoría: TÉCNICO
    
Ticket: "Necesito cambiar el método de pago de mi suscripción"
Categoría: FACTURACIÓN
    
Ticket: "¿Cuál es su política de privacidad de datos?"
Categoría: GENERAL
    
Ticket: "{nuevo_ticket}"
Categoría:"""
    
    print("USANDO EJEMPLOS BALANCEADOS:")
    try:
        response = llm.invoke([HumanMessage(content=prompt_balanceado)])
        resultado = response.content.strip()
        print(f"Clasificación: {resultado}")
        
        # Análisis del balanceo en ejemplos
        ejemplos_tecnico = prompt_balanceado.count("TÉCNICO")
        ejemplos_facturacion = prompt_balanceado.count("FACTURACIÓN")
        ejemplos_general = prompt_balanceado.count("GENERAL")
        
        print(f"\nDistribución de ejemplos:")
        print(f"• TÉCNICO: {ejemplos_tecnico-1} ejemplos")  # -1 porque cuenta el resultado también
        print(f"• FACTURACIÓN: {ejemplos_facturacion-1} ejemplos")
        print(f"• GENERAL: {ejemplos_general-1} ejemplos")
        print(f"✓ Balanceado: 2 ejemplos por categoría")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar balanceo
balanceo_categorias()

=== BALANCEO DE CATEGORÍAS ===
USANDO EJEMPLOS BALANCEADOS:
Clasificación: La categoría correcta para el último ticket es: TÉCNICO

La clasificación final de los tickets es:

 - Ticket: "No puedo acceder a mi cuenta, dice que mi contraseña es incorrecta" - Categoría: TÉCNICO
 - Ticket: "¿Cuándo se procesará mi reembolso del mes pasado?" - Categoría: FACTURACIÓN
 - Ticket: "¿Tienen planes de expandirse a otros países?" - Categoría: GENERAL
 - Ticket: "El botón de exportar datos no funciona en Chrome" - Categoría: TÉCNICO
 - Ticket: "Necesito cambiar el método de pago de mi suscripción" - Categoría: FACTURACIÓN
 - Ticket: "¿Cuál es su política de privacidad de datos?" - Categoría: GENERAL
 - Ticket: "Mi aplicación se cierra inesperadamente cuando intento abrir archivos grandes. ¿Pueden ayudarme?" - Categoría: TÉCNICO

Distribución de ejemplos:
• TÉCNICO: 2 ejemplos
• FACTURACIÓN: 2 ejemplos
• GENERAL: 2 ejemplos
✓ Balanceado: 2 ejemplos por categoría


## Optimización del Número de Ejemplos

¿Cuántos ejemplos necesitamos? Depende de la tarea y complejidad.

In [8]:
# Experimento: 1-shot vs 3-shot vs 5-shot
def optimizar_numero_ejemplos():
    print("=== OPTIMIZACIÓN DEL NÚMERO DE EJEMPLOS ===")
    
    # Tarea: Convertir descripciones casuales a formato técnico
    descripcion = "La página web no carga bien en mi móvil"
    
    # 1-shot
    prompt_1_shot = f"""Convierte descripciones casuales a formato técnico:
    
Casual: "Mi computadora va muy lenta"
Técnico: "Degradación del rendimiento del sistema - posible alto uso de CPU/RAM o fragmentación de disco"
    
Casual: "{descripcion}"
Técnico:"""
    
    # 3-shot
    prompt_3_shot = f"""Convierte descripciones casuales a formato técnico:
    
Casual: "Mi computadora va muy lenta"
Técnico: "Degradación del rendimiento del sistema - posible alto uso de CPU/RAM o fragmentación de disco"
    
Casual: "No puedo enviar emails"
Técnico: "Fallo en servicio SMTP - verificar configuración de servidor de correo saliente"
    
Casual: "La impresora no funciona"
Técnico: "Error de conectividad o driver de dispositivo de impresión - revisar estado de cola y drivers"
    
Casual: "{descripcion}"
Técnico:"""
    
    # 5-shot
    prompt_5_shot = f"""Convierte descripciones casuales a formato técnico:
    
Casual: "Mi computadora va muy lenta"
Técnico: "Degradación del rendimiento del sistema - posible alto uso de CPU/RAM o fragmentación de disco"
    
Casual: "No puedo enviar emails"
Técnico: "Fallo en servicio SMTP - verificar configuración de servidor de correo saliente"
    
Casual: "La impresora no funciona"
Técnico: "Error de conectividad o driver de dispositivo de impresión - revisar estado de cola y drivers"
    
Casual: "El WiFi se desconecta constantemente"
Técnico: "Inestabilidad de conexión inalámbrica - posible interferencia o problema de configuración de red"
    
Casual: "No encuentro mis archivos"
Técnico: "Error de indexación del sistema de archivos - verificar integridad del directorio y permisos"
    
Casual: "{descripcion}"
Técnico:"""
    
    prompts = {
        "1-shot": prompt_1_shot,
        "3-shot": prompt_3_shot,
        "5-shot": prompt_5_shot
    }
    
    for nombre, prompt in prompts.items():
        print(f"\n{nombre.upper()}:")
        print("-" * 20)
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            resultado = response.content.strip()
            print(f"Resultado: {resultado}")
            
            # Métricas básicas
            tokens_prompt = len(prompt.split())
            tecnicidad = sum(1 for palabra in ['TCP', 'HTTP', 'API', 'DNS', 'CSS', 'JavaScript', 'servidor', 'browser', 'responsive'] if palabra.lower() in resultado.lower())
            
            print(f"• Tokens del prompt: ~{tokens_prompt}")
            print(f"• Nivel técnico: {tecnicidad}/9 términos técnicos")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print("\n=== CONCLUSIONES ===")
    print("• 1-shot: Rápido pero puede ser inconsistente")
    print("• 3-shot: Balance entre costo y calidad")
    print("• 5-shot: Más consistente pero más costoso")
    print("• Regla general: 3-5 ejemplos para la mayoría de tareas")

# Ejecutar optimización
optimizar_numero_ejemplos()

=== OPTIMIZACIÓN DEL NÚMERO DE EJEMPLOS ===

1-SHOT:
--------------------
Resultado: Aquí te presento la conversión de la descripción casual a formato técnico:

Casual: "La página web no carga bien en mi móvil"
Técnico: "Error de carga de contenido - posible falla en la comunicación HTTP o problemas de rendimiento de la conexión Wi-Fi/Móvil, falta de optimización de la página web para dispositivos móviles o alto uso de recursos del navegador"

Otras posibles causas técnicas podrían incluir:

- Problemas de configuración de la caché o la memoria cache del navegador
- Uso excesivo de recursos del dispositivo móvil (CPU, RAM, batería)
- Error en la configuración de la red Wi-Fi o móvil
- Problemas de compatibilidad entre el navegador del dispositivo móvil y la página web
- Uso de tecnologías o frameworks obsoletos en la página web que no son compatibles con los dispositivos móviles.
• Tokens del prompt: ~39
• Nivel técnico: 1/9 términos técnicos

3-SHOT:
--------------------
Resultado: Aq

## Casos de Uso Avanzados

In [9]:
# Caso 1: Generación de Código con Patrones Específicos
def few_shot_codigo():
    print("=== FEW-SHOT PARA GENERACIÓN DE CÓDIGO ===")
    
    # Tarea: Generar funciones Python con docstrings específicos
    nueva_funcion = "función que calcule el área de un círculo"
    
    prompt_codigo = f"""Genera funciones Python con docstrings en formato Google Style:
    
Descripción: función que suma dos números
Código:
```python
def sumar(a: float, b: float) -> float:
    \"\"\"Suma dos números y retorna el resultado.
    
    Args:
        a (float): Primer número a sumar.
        b (float): Segundo número a sumar.
        
    Returns:
        float: La suma de a y b.
        
    Example:
        >>> sumar(3.5, 2.1)
        5.6
    \"\"\"
    return a + b
```
    
Descripción: función que encuentra el máximo en una lista
Código:
```python
def encontrar_maximo(numeros: list[float]) -> float:
    \"\"\"Encuentra el valor máximo en una lista de números.
    
    Args:
        numeros (list[float]): Lista de números a evaluar.
        
    Returns:
        float: El valor máximo de la lista.
        
    Raises:
        ValueError: Si la lista está vacía.
        
    Example:
        >>> encontrar_maximo([1.5, 3.2, 2.1])
        3.2
    \"\"\"
    if not numeros:
        raise ValueError("La lista no puede estar vacía")
    return max(numeros)
```
    
Descripción: {nueva_funcion}
Código:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_codigo)])
        print("FUNCIÓN GENERADA:")
        print(response.content)
        
        # Análisis de calidad
        codigo = response.content
        tiene_type_hints = ': ' in codigo and '->' in codigo
        tiene_docstring = '"""' in codigo
        tiene_args = 'Args:' in codigo
        tiene_returns = 'Returns:' in codigo
        tiene_example = 'Example:' in codigo
        
        print("\n=== ANÁLISIS DE CALIDAD ===")
        print(f"✓ Type hints: {'Sí' if tiene_type_hints else 'No'}")
        print(f"✓ Docstring: {'Sí' if tiene_docstring else 'No'}")
        print(f"✓ Args section: {'Sí' if tiene_args else 'No'}")
        print(f"✓ Returns section: {'Sí' if tiene_returns else 'No'}")
        print(f"✓ Example: {'Sí' if tiene_example else 'No'}")
        
        calidad = sum([tiene_type_hints, tiene_docstring, tiene_args, tiene_returns, tiene_example])
        print(f"\nPuntuación: {calidad}/5 - {'Excelente' if calidad >= 4 else 'Bueno' if calidad >= 3 else 'Necesita mejoras'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar generación de código
few_shot_codigo()

=== FEW-SHOT PARA GENERACIÓN DE CÓDIGO ===
FUNCIÓN GENERADA:
Aquí te dejo las funciones con docstrings en formato Google Style:

```python
import math

def sumar(a: float, b: float) -> float:
    """Suma dos números enteros o floats y retorna el resultado.

    Args:
        a (float): Primer número a sumar.
        b (float): Segundo número a sumar.

    Returns:
        float: La suma de a y b.

    Example:
        >>> sumar(3.5, 2.1)
        5.6
    """
    return a + b

def encontrar_maximo(numeros: list[float]) -> float:
    """Encuentra el valor máximo en una lista de números.

    Args:
        numeros (list[float]): Lista de números a evaluar.

    Returns:
        float: El valor máximo de la lista.

    Raises:
        ValueError: Si la lista está vacía.

    Example:
        >>> encontrar_maximo([1.5, 3.2, 2.1])
        3.2
    """
    if not numeros:
        raise ValueError("La lista no puede estar vacía")
    return max(numeros)

def calcular_area_circulo(radio: float) -

In [10]:
# Caso 2: Transformación de Datos Estructurados
def few_shot_transformacion_datos():
    print("=== FEW-SHOT PARA TRANSFORMACIÓN DE DATOS ===")
    
    # Tarea: Convertir datos de empleados a diferentes formatos
    nuevo_empleado = "Carlos Mendez, Desarrollador Backend, 5 años experiencia, especialista en Python y PostgreSQL"
    
    prompt_transformacion = f"""Convierte descripciones de empleados a formato JSON estructurado:
    
Descripción: "Ana López, Diseñadora UX/UI, 3 años experiencia, experta en Figma y Adobe XD"
JSON:
{{
  "nombre": "Ana López",
  "puesto": "Diseñadora UX/UI",
  "experiencia_años": 3,
  "habilidades": ["Figma", "Adobe XD"],
  "departamento": "Diseño",
  "nivel": "Mid"
}}
    
Descripción: "Miguel Torres, QA Engineer Senior, 7 años experiencia, especializado en Selenium y Jest"
JSON:
{{
  "nombre": "Miguel Torres",
  "puesto": "QA Engineer Senior",
  "experiencia_años": 7,
  "habilidades": ["Selenium", "Jest"],
  "departamento": "Calidad",
  "nivel": "Senior"
}}
    
Descripción: "Laura García, Data Scientist, 2 años experiencia, conocimientos en Python y TensorFlow"
JSON:
{{
  "nombre": "Laura García",
  "puesto": "Data Scientist",
  "experiencia_años": 2,
  "habilidades": ["Python", "TensorFlow"],
  "departamento": "Datos",
  "nivel": "Junior"
}}
    
Descripción: "{nuevo_empleado}"
JSON:
    Tu resultados debe estar en el siguiente formato:
    {{
      "nombre": "Laura García",
      "puesto": "Data Scientist",
      "experiencia_años": 2,
      "habilidades": ["Python", "TensorFlow"],
      "departamento": "Datos",
      "nivel": "Junior"
    }}
        Tu resultados siempre debe ser un JSON. No pongas ninguna palabra antes, ni despues, solo el JSON. No pongas un formato asi:
    ""json{{
  "nombre": "Laura García",
  "puesto": "Data Scientist",
  "experiencia_años": 2,
  "habilidades": ["Python", "TensorFlow"],
  "departamento": "Datos",
  "nivel": "Junior"
}}""
    NUNCA pongas la palabra JSON. 
    Responde ÚNICAMENTE con JSON válido. Los valores de texto DEBEN estar entre comillas dobles.
    Dame solo el JSON del nuevo mensaje, no los demas. 

"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_transformacion)])
        print("TRANSFORMACIÓN REALIZADA:")
        print(response.content)
        
        # Validar JSON y estructura
        try:
            data = json.loads(response.content.strip())
            campos_esperados = ["nombre", "puesto", "experiencia_años", "habilidades", "departamento", "nivel"]
            campos_presentes = list(data.keys())
            
            print("\n=== VALIDACIÓN ===")
            print(f"✓ JSON válido: Sí")
            print(f"✓ Campos esperados: {len(campos_esperados)}")
            print(f"✓ Campos presentes: {len(campos_presentes)}")
            print(f"✓ Estructura completa: {'Sí' if set(campos_esperados).issubset(set(campos_presentes)) else 'No'}")
            
            # Validar tipos de datos
            tipo_experiencia = type(data.get('experiencia_años')).__name__
            tipo_habilidades = type(data.get('habilidades')).__name__
            
            print(f"✓ Experiencia es número: {'Sí' if tipo_experiencia == 'int' else 'No'}")
            print(f"✓ Habilidades es lista: {'Sí' if tipo_habilidades == 'list' else 'No'}")
            
        except json.JSONDecodeError:
            print("\n✗ JSON inválido - formato inconsistente")
    
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar transformación
few_shot_transformacion_datos()

=== FEW-SHOT PARA TRANSFORMACIÓN DE DATOS ===
TRANSFORMACIÓN REALIZADA:
{
  "nombre": "Carlos Mendez",
  "puesto": "Desarrollador Backend",
  "experiencia_años": 5,
  "habilidades": ["Python", "PostgreSQL"],
  "departamento": "Desarrollo",
  "nivel": "Mid"
}

=== VALIDACIÓN ===
✓ JSON válido: Sí
✓ Campos esperados: 6
✓ Campos presentes: 6
✓ Estructura completa: Sí
✓ Experiencia es número: Sí
✓ Habilidades es lista: Sí


## Técnicas Avanzadas de Few-Shot

In [11]:
# Técnica 1: Few-Shot con Explicaciones
def few_shot_con_explicaciones():
    print("=== FEW-SHOT CON EXPLICACIONES ===")
    
    # Tarea: Análisis de código SQL
    nuevo_sql = "SELECT COUNT(*) FROM users WHERE age > 25 AND status = 'active'"
    
    prompt_explicado = f"""Analiza queries SQL y explica qué hacen:
    
SQL: "SELECT name, email FROM employees WHERE department = 'IT'"
Explicación: Esta query selecciona los nombres y emails de todos los empleados que trabajan en el departamento de IT. Usa un filtro WHERE para limitar los resultados solo a empleados de IT.
Complejidad: Básica
    
SQL: "SELECT d.name, COUNT(e.id) FROM departments d LEFT JOIN employees e ON d.id = e.dept_id GROUP BY d.name"
Explicación: Esta query cuenta cuántos empleados hay en cada departamento. Usa LEFT JOIN para incluir departamentos sin empleados (mostrarían 0) y GROUP BY para agrupar los resultados por nombre de departamento.
Complejidad: Intermedia
    
SQL: "SELECT * FROM products WHERE price BETWEEN 100 AND 500 ORDER BY price DESC"
Explicación: Esta query obtiene todos los productos con precios entre 100 y 500, ordenados de mayor a menor precio. BETWEEN incluye ambos valores límite (100 y 500).
Complejidad: Básica
    
SQL: "{nuevo_sql}"
Explicación:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_explicado)])
        print("ANÁLISIS GENERADO:")
        print(response.content)
        
        # Verificar si siguió el patrón
        resultado = response.content
        tiene_explicacion = len(resultado) > 50
        menciona_complejidad = 'complejidad' in resultado.lower()
        explica_clausulas = any(palabra in resultado.lower() for palabra in ['where', 'count', 'select'])
        
        print("\n=== ANÁLISIS DEL PATRÓN ===")
        print(f"✓ Explicación detallada: {'Sí' if tiene_explicacion else 'No'}")
        print(f"✓ Menciona complejidad: {'Sí' if menciona_complejidad else 'No'}")
        print(f"✓ Explica cláusulas SQL: {'Sí' if explica_clausulas else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar few-shot con explicaciones
few_shot_con_explicaciones()

=== FEW-SHOT CON EXPLICACIONES ===
ANÁLISIS GENERADO:
**Análisis de consultas SQL**

### 1. `SELECT name, email FROM employees WHERE department = 'IT'`

Esta consulta selecciona los campos `name` y `email` de la tabla `employees` donde el valor del campo `department` sea `'IT'`. La cláusula `WHERE` se utiliza para filtrar los resultados y solo mostrar los empleados que trabajan en el departamento de IT.

**Complejidad:** Básica

### 2. `SELECT d.name, COUNT(e.id) FROM departments d LEFT JOIN employees e ON d.id = e.dept_id GROUP BY d.name`

Esta consulta cuenta el número de empleados en cada departamento. Se utiliza un `LEFT JOIN` para incluir departamentos que no tienen empleados (se mostrarían con 0 empleados). La cláusula `GROUP BY` se utiliza para agrupar los resultados por el nombre del departamento.

* `d.name`: selecciona el nombre del departamento.
* `COUNT(e.id)`: cuenta el número de empleados en cada departamento.
* `LEFT JOIN`: combina la tabla `departments` con la tabla `em

In [12]:
# Técnica 2: Few-Shot Progresivo (Aumentando Complejidad)
def few_shot_progresivo():
    print("=== FEW-SHOT PROGRESIVO ===")
    
    # Tarea: Generar expresiones regulares
    nueva_tarea = "expresión regular para validar números de teléfono españoles (+34 XXX XXX XXX)"
    
    prompt_progresivo = f"""Genera expresiones regulares con explicación:
    
Tarea: validar email básico
Regex: ^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{{2,}}$
Explicación: Valida formato básico de email con caracteres permitidos antes y después del @, y dominio con al menos 2 caracteres.
Complejidad: ⭐⭐
    
Tarea: extraer URLs de texto
Regex: https?://(?:[-\w.])+(?::[0-9]+)?(?:/(?:[\w/_.])*(?:\?(?:[\w&=%.])*)?(?:#(?:[\w.])*)?)?)
Explicación: Captura URLs HTTP/HTTPS con dominio, puerto opcional, path, query parameters y fragmentos.
Complejidad: ⭐⭐⭐⭐
    
Tarea: validar fecha formato DD/MM/YYYY
Regex: ^(?:(?:31(\/|-|\.)(?:0?[13578]|1[02]))\1|(?:(?:29|30)(\/|-|\.)(?:0?[13-9]|1[0-2])\2))(?:(?:1[6-9]|[2-9]\d)?\d{{2}})$|^(?:29(\/|-|\.)0?2\3(?:(?:(?:1[6-9]|[2-9]\d)?(?:0[48]|[2468][048]|[13579][26])|(?:(?:16|[2468][048]|[3579][26])00))))$|^(?:0?[1-9]|1\d|2[0-8])(\/|-|\.)(?:(?:0?[1-9])|(?:1[0-2]))\4(?:(?:1[6-9]|[2-9]\d)?\d{{2}})$
Explicación: Valida fechas considerando años bisiestos, días válidos por mes, y diferentes separadores. Muy compleja pero precisa.
Complejidad: ⭐⭐⭐⭐⭐
    
Tarea: {nueva_tarea}
Regex:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_progresivo)])
        print("EXPRESIÓN REGULAR GENERADA:")
        print(response.content)
        
        # Analizar complejidad
        resultado = response.content
        tiene_regex = bool(re.search(r'[\[\](){}.*+?^$|\\]', resultado))
        tiene_explicacion = 'Explicación:' in resultado
        tiene_complejidad = '⭐' in resultado or 'Complejidad:' in resultado
        
        print("\n=== ANÁLISIS ===")
        print(f"✓ Contiene regex: {'Sí' if tiene_regex else 'No'}")
        print(f"✓ Incluye explicación: {'Sí' if tiene_explicacion else 'No'}")
        print(f"✓ Indica complejidad: {'Sí' if tiene_complejidad else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar few-shot progresivo
import re
few_shot_progresivo()

=== FEW-SHOT PROGRESIVO ===


<>:26: SyntaxWarning: invalid escape sequence '\.'
<>:26: SyntaxWarning: invalid escape sequence '\w'
<>:26: SyntaxWarning: invalid escape sequence '\/'
<>:26: SyntaxWarning: invalid escape sequence '\.'
<>:26: SyntaxWarning: invalid escape sequence '\w'
<>:26: SyntaxWarning: invalid escape sequence '\/'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13668\1103443465.py:26: SyntaxWarning: invalid escape sequence '\.'
  Regex:"""
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13668\1103443465.py:26: SyntaxWarning: invalid escape sequence '\w'
  Regex:"""
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13668\1103443465.py:26: SyntaxWarning: invalid escape sequence '\/'
  Regex:"""


EXPRESIÓN REGULAR GENERADA:
**Tarea: validar email básico**

La expresión regular para validar un email básico es:

`^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$`

Explicación:

- `^` indica el comienzo del texto a validar.
- `[a-zA-Z0-9._%+-]+` permite cualquier carácter alfanumérico, punto, guión bajo, porcentaje, signo de más o menos. El `+` indica que puede aparecer 1 o más veces.
- `@` es el carácter que separa el nombre de usuario del dominio.
- `[a-zA-Z0-9.-]+` permite cualquier carácter alfanumérico, punto o guión. De nuevo, el `+` indica que puede aparecer 1 o más veces.
- `\.` es un punto, el carácter que separa el dominio del dominio de nivel superior.
- `[a-zA-Z]{2,}` permite cualquier carácter alfanumérico en el dominio de nivel superior, con un mínimo de 2 caracteres.
- `$` indica el final del texto a validar.

Complejidad: ⭐⭐

**Tarea: extraer URLs de texto**

La expresión regular para extraer URLs de texto es:

`https?://(?:[-\w.])+(?::[0-9]+)?(?:/(?:[\w/_.])*(?:\?(

## Mejores Prácticas y Consideraciones

### ✅ Cuándo Usar Few-Shot:
- Formato de salida específico requerido
- Tareas con patrones complejos
- Cuando la consistencia es crítica
- Dominios especializados con ejemplos disponibles

### 🎯 Selección de Ejemplos:
1. **Diversidad**: Cubrir diferentes variaciones de la tarea
2. **Calidad**: Ejemplos perfectos que seguir
3. **Balance**: Representar todas las categorías equitativamente
4. **Claridad**: Ejemplos inequívocos y bien estructurados

### ⚠️ Limitaciones:
- **Costo**: Más tokens = mayor costo
- **Contexto**: Límite de ventana de contexto
- **Sesgos**: Los ejemplos pueden introducir sesgos
- **Overfitting**: Puede ser demasiado rígido

In [13]:
# Ejercicio final: Diseña tu propio few-shot prompt
def ejercicio_few_shot():
    print("=== EJERCICIO: DISEÑA TU FEW-SHOT PROMPT ===")
    print("\nTarea: Crear un sistema que convierta descripciones de bugs en tickets estructurados")
    print("\nRequisitos:")
    print("- Extraer: título, descripción, severidad, componente afectado")
    print("- Formato JSON consistente")
    print("- Clasificar severidad: Alta, Media, Baja")
    print("- Identificar componente: Frontend, Backend, Database, API")
    
    # Ejemplo de bug description para procesar
    bug_ejemplo = """Cuando hago clic en el botón de login después de ingresar credenciales válidas, 
    la página se queda en blanco y no pasa nada. Esto pasa solo en Chrome. El error aparece 
    en la consola como 'TypeError: Cannot read property of undefined'."""
    
    print(f"\nBug a procesar: {bug_ejemplo}")
    print("\nDiseña un few-shot prompt con 3 ejemplos que cubran:")
    print("1. Bug de frontend")
    print("2. Bug de backend/API")
    print("3. Bug de base de datos")
    
    # Template para que el estudiante complete
    template = """
    # TU PROMPT AQUÍ:
    
    Convierte descripciones de bugs a tickets estructurados en JSON:
    
    Bug: "[EJEMPLO 1 - Frontend]"
    Ticket: {
        "titulo": "...",
        "descripcion": "...",
        "severidad": "...",
        "componente": "..."
    }
    
    Bug: "[EJEMPLO 2 - Backend/API]"
    Ticket: {...}
    
    Bug: "[EJEMPLO 3 - Database]"
    Ticket: {...}
    
    Bug: "{bug_description}"
    Ticket:
    """
    
    print(template)
    
    # Prompt de ejemplo bien diseñado
    prompt_ejemplo = f"""Convierte descripciones de bugs a tickets estructurados:
    
Bug: "El botón de buscar no responde cuando hay caracteres especiales en el input. Sale error 'Invalid character' en el navegador."
Ticket: {{
    "titulo": "Botón de búsqueda falla con caracteres especiales",
    "descripcion": "Error de validación en frontend al procesar caracteres especiales en campo de búsqueda",
    "severidad": "Media",
    "componente": "Frontend"
}}
    
Bug: "La API devuelve 500 cuando se hace POST a /users con email duplicado. Debería devolver 400 con mensaje claro."
Ticket: {{
    "titulo": "API retorna código de error incorrecto para email duplicado",
    "descripcion": "Endpoint POST /users devuelve 500 en lugar de 400 para email duplicado",
    "severidad": "Alta",
    "componente": "API"
}}
    
Bug: "Los reportes no cargan desde ayer, timeout en queries que normalmente son rápidas. Posible problema de índices."
Ticket: {{
    "titulo": "Degradación de performance en queries de reportes",
    "descripcion": "Timeout en queries de reportes sugiere problema de optimización o índices faltantes",
    "severidad": "Alta",
    "componente": "Database"
}}
    
Bug: "{bug_ejemplo}"
Ticket:"""
    
    print("\n=== PROMPT DE REFERENCIA ===")
    print(prompt_ejemplo)
    
    # Ejecutar el prompt ejemplo
    print("\n=== RESULTADO DEL PROMPT DE REFERENCIA ===")
    try:
        response = llm.invoke([HumanMessage(content=prompt_ejemplo)])
        print(response.content)
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejercicio
ejercicio_few_shot()

=== EJERCICIO: DISEÑA TU FEW-SHOT PROMPT ===

Tarea: Crear un sistema que convierta descripciones de bugs en tickets estructurados

Requisitos:
- Extraer: título, descripción, severidad, componente afectado
- Formato JSON consistente
- Clasificar severidad: Alta, Media, Baja
- Identificar componente: Frontend, Backend, Database, API

Bug a procesar: Cuando hago clic en el botón de login después de ingresar credenciales válidas, 
    la página se queda en blanco y no pasa nada. Esto pasa solo en Chrome. El error aparece 
    en la consola como 'TypeError: Cannot read property of undefined'.

Diseña un few-shot prompt con 3 ejemplos que cubran:
1. Bug de frontend
2. Bug de backend/API
3. Bug de base de datos

    # TU PROMPT AQUÍ:
    
    Convierte descripciones de bugs a tickets estructurados en JSON:
    
    Bug: "[EJEMPLO 1 - Frontend]"
    Ticket: {
        "titulo": "...",
        "descripcion": "...",
        "severidad": "...",
        "componente": "..."
    }
    
    Bug: "[E

## Conceptos Clave Aprendidos

1. **Few-shot prompting** mejora consistencia y control de formato
2. **Selección de ejemplos** es crucial para el éxito
3. **Balance de categorías** evita sesgos en clasificación
4. **3-5 ejemplos** es generalmente óptimo para la mayoría de tareas
5. **Costo vs. calidad** debe considerarse en la implementación

## Próximos Pasos

En el siguiente notebook exploraremos **Chain-of-Thought Prompting**, una técnica que hace que el modelo "piense en voz alta" y muestre su razonamiento paso a paso, especialmente útil para problemas complejos y matemáticos.

### Para Practicar:
1. Imaginemos que tenemos un schema de base de datos transaccional, donde se encuentran las siguientes tablas: clientes, transacciones, productos, tiendas y canal de un supermercado en Chile. 

El esquema es el siguiente:

CREATE TABLE clientes (
    cliente_id SERIAL PRIMARY KEY,
    rut VARCHAR(12) UNIQUE,
    nombre VARCHAR(100) NOT NULL,
    apellido VARCHAR(100) NOT NULL,
    email VARCHAR(150),
    telefono VARCHAR(20),
    fecha_nacimiento DATE,
    fecha_registro TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE canales (
    canal_id SERIAL PRIMARY KEY,
    nombre VARCHAR(50) NOT NULL,
    descripcion TEXT
);

CREATE TABLE tiendas (
    tienda_id SERIAL PRIMARY KEY,
    nombre VARCHAR(150) NOT NULL,
    region VARCHAR(100),
    ciudad VARCHAR(100),
    direccion TEXT
);

CREATE TABLE productos (
    producto_id SERIAL PRIMARY KEY,
    nombre VARCHAR(200) NOT NULL,
    categoria VARCHAR(100),
    marca VARCHAR(100),
    precio NUMERIC(10,2)
);

CREATE TABLE transacciones (
    transaccion_id SERIAL PRIMARY KEY,
    cliente_id INT,
    tienda_id INT,
    canal_id INT,
    fecha TIMESTAMP NOT NULL,
    monto_total NUMERIC(12,2),
    FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id),
    FOREIGN KEY (tienda_id) REFERENCES tiendas(tienda_id),
    FOREIGN KEY (canal_id) REFERENCES canales(canal_id)
);


2. Construye un prompt de sistema con ejemplo de pregunta y SQL que sirve para construir un analista de datos (SQL). El objetivo es que sea capaz de en base a una pregunta en lenguaje transformala a una consulta en SQL. Como lo hemos visto, dale tres ejemplos de posibles preguntas y SQL para guiarlo en la construccion de la respuesta.

In [14]:
schema_ddbb= """CREATE TABLE clientes (
    cliente_id SERIAL PRIMARY KEY,
    rut VARCHAR(12) UNIQUE,
    nombre VARCHAR(100) NOT NULL,
    apellido VARCHAR(100) NOT NULL,
    email VARCHAR(150),
    telefono VARCHAR(20),
    fecha_nacimiento DATE,
    fecha_registro TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE canales (
    canal_id SERIAL PRIMARY KEY,
    nombre VARCHAR(50) NOT NULL,
    descripcion TEXT
);

CREATE TABLE tiendas (
    tienda_id SERIAL PRIMARY KEY,
    nombre VARCHAR(150) NOT NULL,
    region VARCHAR(100),
    ciudad VARCHAR(100),
    direccion TEXT
);

CREATE TABLE productos (
    producto_id SERIAL PRIMARY KEY,
    nombre VARCHAR(200) NOT NULL,
    categoria VARCHAR(100),
    marca VARCHAR(100),
    precio NUMERIC(10,2)
);

CREATE TABLE transacciones (
    transaccion_id SERIAL PRIMARY KEY,
    cliente_id INT,
    tienda_id INT,
    canal_id INT,
    fecha TIMESTAMP NOT NULL,
    monto_total NUMERIC(12,2),
    FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id),
    FOREIGN KEY (tienda_id) REFERENCES tiendas(tienda_id),
    FOREIGN KEY (canal_id) REFERENCES canales(canal_id)
);"""

In [31]:
prompt_system = """
ROLE
You are an expert in SQL, specialized in PostgreSQL syntax.

MISSION
Transform the user's question into a valid PostgreSQL query based on the provided schema.

INSTRUCTIONS:
1- Detect the information the user is asking based on the schema.
2- Write the query using strictly PostgreSQL syntax:
   - Use DATE_TRUNC, INTERVAL, CURRENT_DATE for date operations.
   - Use LIMIT for row limiting (not TOP).
   - Use ILIKE for case-insensitive text search.
   - Cast types explicitly when needed (e.g., ::DATE, ::VARCHAR).
3- If you are not 100% sure about the query, ask the user to clarify before answering.

DATABASE:
- Engine: PostgreSQL 15+
- Schema: {}

OUTPUT:
Return only the SQL query. No explanations, no markdown, no extra text.

RESTRICCIONES:
- Debes siempre responder respecto a la informacion de la base. Si es una pregunta 
que no se relaciona con eso, comenta que tu funcion principal es ser un agente SQL.
No inventes nada.


EXAMPLES.
USER: ¿Cuantas compras hizo el cliente rut 18.000.111 la semana pasada?
ANSWER: SELECT COUNT(DISTINCT transaccion_id)
        FROM transacciones
        LEFT JOIN clientes ON transacciones.cliente_id = clientes.cliente_id
        WHERE rut = '18.000.111'
          AND fecha >= DATE_TRUNC('week', CURRENT_DATE) - INTERVAL '7 days'
          AND fecha <  DATE_TRUNC('week', CURRENT_DATE)

USER: ¿Cuáles son los clientes que más han gastado en total? TOP 5.
ANSWER: SELECT c.rut, c.nombre, c.apellido, SUM(tr.monto_total) AS gasto_total
        FROM transacciones tr
        LEFT JOIN clientes c ON tr.cliente_id = c.cliente_id
        GROUP BY c.rut, c.nombre, c.apellido
        ORDER BY gasto_total DESC
        LIMIT 5

USER: ¿Cuánto vendió cada tienda en enero de 2026?
ANSWER: SELECT t.nombre AS tienda, SUM(tr.monto_total) AS total_ventas
        FROM transacciones tr
        LEFT JOIN tiendas t ON tr.tienda_id = t.tienda_id
        WHERE tr.fecha >= '2026-01-01'::DATE
          AND tr.fecha <  '2026-02-01'::DATE
        GROUP BY t.nombre
        ORDER BY total_ventas DESC

""".format(schema_ddbb)


In [32]:
prompt_system

"\nROLE\nYou are an expert in SQL, specialized in PostgreSQL syntax.\n\nMISSION\nTransform the user's question into a valid PostgreSQL query based on the provided schema.\n\nINSTRUCTIONS:\n1- Detect the information the user is asking based on the schema.\n2- Write the query using strictly PostgreSQL syntax:\n   - Use DATE_TRUNC, INTERVAL, CURRENT_DATE for date operations.\n   - Use LIMIT for row limiting (not TOP).\n   - Use ILIKE for case-insensitive text search.\n   - Cast types explicitly when needed (e.g., ::DATE, ::VARCHAR).\n3- If you are not 100% sure about the query, ask the user to clarify before answering.\n\nDATABASE:\n- Engine: PostgreSQL 15+\n- Schema: CREATE TABLE clientes (\n    cliente_id SERIAL PRIMARY KEY,\n    rut VARCHAR(12) UNIQUE,\n    nombre VARCHAR(100) NOT NULL,\n    apellido VARCHAR(100) NOT NULL,\n    email VARCHAR(150),\n    telefono VARCHAR(20),\n    fecha_nacimiento DATE,\n    fecha_registro TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n);\n\nCREATE TABLE canales (

In [39]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

def sql_agent(system_prompt, messages_history):

    llm = ChatGroq(
        api_key=os.getenv("GROQ_API_KEY"),
        model_name="llama-3.1-8b-instant",
        temperature=0
    )

    messages = [
        SystemMessage(system_prompt),
#        HumanMessage("Write a haiku about spring"),
#        AIMessage("Cherry blossoms bloom...")
    ]
    messages = messages  + messages_history
    response = llm.invoke(messages)
    return response.content


messages_history = [HumanMessage("cual es el clientes con mas compras")]

sql_agent(prompt_system, messages_history)



'SELECT c.rut, c.nombre, c.apellido, COUNT(DISTINCT tr.transaccion_id) AS num_compras\nFROM transacciones tr\nLEFT JOIN clientes c ON tr.cliente_id = c.cliente_id\nGROUP BY c.rut, c.nombre, c.apellido\nORDER BY num_compras DESC\nLIMIT 1'

In [43]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ── SQL agent ────────────────────────────────────────────────────────────────
def sql_agent(system_prompt, messages_history):
    llm = ChatGroq(
        api_key=os.getenv("GROQ_API_KEY"),
        model_name="llama-3.1-8b-instant",
        temperature=0
    )
    messages = [SystemMessage(content=system_prompt)] + messages_history
    response = llm.invoke(messages)
    return response.content

# ── System prompt  ─────────────────────────────────────
prompt_system = prompt_system

# ── Conversation state ────────────────────────────────────────────────────────
messages_history = []

# ── Widgets ───────────────────────────────────────────────────────────────────
text_input  = widgets.Text(
    placeholder="Escribe tu pregunta sobre los datos...",
    layout=widgets.Layout(width="75%")
)
send_btn    = widgets.Button(description="Enviar", button_style="primary")
clear_btn   = widgets.Button(description="Limpiar chat", button_style="warning")
chat_output = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", padding="10px", min_height="200px", overflow_y="auto"
))

def render_history():
    """Redibuja todo el historial en el output widget."""
    with chat_output:
        clear_output(wait=True)
        for msg in messages_history:
            if isinstance(msg, HumanMessage):
                print(f"🧑 Tú: {msg.content}")
            elif isinstance(msg, AIMessage):
                print(f"🤖 SQL:\n{msg.content}")
            print()

def on_send(b):
    question = text_input.value.strip()
    if not question:
        return
    text_input.value = ""

    messages_history.append(HumanMessage(content=question))
    render_history()

    try:
        answer = sql_agent(prompt_system, messages_history)
        messages_history.append(AIMessage(content=answer))
    except Exception as e:
        answer = f"[Error: {e}]"
        messages_history.append(AIMessage(content=answer))

    render_history()

def on_clear(b):
    messages_history.clear()
    with chat_output:
        clear_output()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)   # Enter key also sends

# ── Layout ────────────────────────────────────────────────────────────────────
display(
    widgets.HTML("<h3>💬 SQL Agent — Analista de Datos Interactivo</h3>"),
    chat_output,
    widgets.HBox([text_input, send_btn, clear_btn])
)

HTML(value='<h3>💬 SQL Agent — Analista de Datos Interactivo</h3>')

Output(layout=Layout(border='1px solid #ccc', min_height='200px', overflow_y='auto', padding='10px'))